# GEANT h96 Colab — interrupted seed-42 recovery

This is an **export-only** recovery workflow. It does not resume, relaunch, train, load a checkpoint, or read dataset arrays. It validates the reported incomplete attempt and exports its manifest plus any already-persisted launcher log to a new ZIP in a user-selected persistent location. It writes a flushed recovery journal to that location before and during packaging, so a packaging interruption leaves a diagnostic journal there.

The trainer writes `status: incomplete` before training, updates `epoch_history` only after an epoch finishes, and catches Python exceptions to try to write `failed`. A kill, kernel/runtime loss, machine reset, or termination before the handler runs can therefore leave `failure: null` and an empty history. The observation does not identify which cause occurred. The reported ~551 MB peak is below the 2 GiB process ceiling, so the available evidence does not point to that process limit; it cannot rule out a host/cgroup event, external termination, or runtime loss.

The training runner itself emits no per-batch progress and commits history only at epoch boundaries. A parent notebook can persist received output and periodic manifest/resource snapshots to Drive, but `fsync` is not a guarantee of Google Drive synchronization or survival of abrupt VM loss. This notebook only exports the already-interrupted attempt; it never retries it. A future training attempt needs a separately reviewed launcher/output namespace and explicit authorization.

In [ ]:
import hashlib
import json
import os
import stat
import time
import uuid
import zipfile
from pathlib import Path

SESSION_ROOT = Path('/content/geant_h96_colab_prepare_v1_e015c2e49eb04dfba84124c39af79a38')
REPOSITORY_ROOT = SESSION_ROOT / 'approved_source_snapshot'
OUTPUT_ROOT = REPOSITORY_ROOT / 'data/processed/geant_horizon96_streaming_experiment_v2_colab_memv3'
RUN_DIR = OUTPUT_ROOT / 'h96_topology_seed_42'
ALLOWED_RUN_FILES = {'run.json'}

def _checked_existing_path(path, *, directory=None):
    path = Path(os.path.abspath(path))
    current = Path(path.anchor)
    for component in path.parts[1:]:
        current = current / component
        info = current.lstat()
        if current.is_symlink() or stat.S_ISLNK(info.st_mode):
            raise RuntimeError(f'Refusing symlink/path redirection: {current}')
    if directory is True and not path.is_dir():
        raise RuntimeError(f'Expected directory: {path}')
    if directory is False and not path.is_file():
        raise RuntimeError(f'Expected regular file: {path}')
    return path

def _validate_interrupted_attempt(run_dir):
    run_dir = _checked_existing_path(run_dir, directory=True)
    if run_dir != RUN_DIR:
        raise RuntimeError(f'Recovery is restricted to the known seed-42 attempt: {RUN_DIR}')
    if {item.name for item in run_dir.iterdir()} != ALLOWED_RUN_FILES:
        raise RuntimeError('Attempt contents differ from the reported manifest-only incomplete state; stop for review')
    manifest_path = _checked_existing_path(run_dir / 'run.json', directory=False)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('status') != 'incomplete' or manifest.get('failure') is not None
            or manifest.get('epoch_history', []) != [] or manifest.get('seed') != 42
            or manifest.get('adjacency_mode') != 'topology'
            or manifest.get('protocol_id') != 'geant-horizon96-only-candidate-v2-batch4'
            or manifest.get('test_access', {}).get('permitted') is not False):
        raise RuntimeError('Manifest does not match the expected incomplete seed-42 topology attempt')
    config = manifest.get('configuration', {})
    expected_config = {
        'history_length': 672, 'horizons': [96], 'batch_size': 4,
        'epochs': 20, 'patience': 4, 'learning_rate': 0.005,
        'optimizer': 'Adam', 'hidden_size': 16, 'ewma_alpha': 0.8, 'ewma_steps': 5,
        'residual_loss': 'SmoothL1(beta=0.001) in original utilization-ratio units',
    }
    differences = {key: (value, config.get(key)) for key, value in expected_config.items() if config.get(key) != value}
    if differences:
        raise RuntimeError(f'Attempt scientific configuration differs from the approved h96 protocol: {differences}')
    policy = config.get('linux_resource_policy', {})
    if (policy.get('policy_id') != 'colab-linux-process-memory-v3-visible-cgroup'
            or policy.get('process_peak_limit_bytes') != 2 * 1024**3
            or policy.get('in_run_os_available_reserve_bytes') != 384 * 1024**2
            or policy.get('in_run_cgroup_headroom_reserve_bytes') != 384 * 1024**2
            or policy.get('minimum_free_disk_bytes') != 512 * 1024**2):
        raise RuntimeError('Attempt manifest does not record the approved Colab v3 resource policy')
    return manifest_path, manifest

def _append_progress_durably(log_paths, line):
    payload = str(line)
    if not payload.endswith('\n'):
        payload += '\n'
    for raw_path in log_paths:
        path = Path(raw_path)
        if path.exists() or path.is_symlink():
            _checked_existing_path(path, directory=False)
        else:
            _checked_existing_path(path.parent, directory=True)
        with path.open('a', encoding='utf-8', newline='') as stream:
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())

def _assert_no_active_training_process(repository_root, proc_root=Path('/proc')):
    proc_root = Path(proc_root)
    if not proc_root.is_dir():
        raise RuntimeError(f'Cannot verify that training is inactive: process table unavailable at {proc_root}')
    repository_text = str(Path(repository_root).resolve())
    signatures = ('horizon96_streaming_experiment_v2.py', 'colab_h96_v1/launch.py')
    for entry in proc_root.iterdir():
        if not entry.name.isdigit():
            continue
        try:
            command = (entry / 'cmdline').read_bytes().replace(b'\x00', b' ').decode('utf-8', errors='replace')
        except (FileNotFoundError, ProcessLookupError):
            continue
        except PermissionError as exc:
            raise RuntimeError(f'Cannot verify process {entry.name}; refusing recovery/export') from exc
        if repository_text in command and any(signature in command for signature in signatures):
            raise RuntimeError(f'Possible GEANT training process is still active (pid {entry.name}); do not export or retry yet')

def _sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _stream_child_output(process, log_paths, emit=print):
    if process.stdout is None:
        raise RuntimeError('Child process has no captured stdout')
    for line in process.stdout:
        _append_progress_durably(log_paths, line)
        emit(line, end='', flush=True)
    return process.wait()

def _export_incomplete_attempt(run_dir, session_root, export_directory, *, process_check=_assert_no_active_training_process):
    session_root = _checked_existing_path(session_root, directory=True)
    export_directory = _checked_existing_path(export_directory, directory=True)
    process_check(REPOSITORY_ROOT)
    journal_path = export_directory / f'h96_seed42_recovery_progress_{uuid.uuid4().hex}.jsonl'
    def journal(event, **details):
        row = {'time_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(), 'event': event, **details}
        _append_progress_durably([journal_path], json.dumps(row, sort_keys=True))
    journal('recovery_started', run_directory=str(run_dir), export_directory=str(export_directory))
    try:
        manifest_path, manifest = _validate_interrupted_attempt(run_dir)
        journal('manifest_validated', manifest_sha256=_sha256_path(manifest_path), status=manifest['status'])
    except BaseException as exc:
        journal('recovery_validation_failed', error=f'{type(exc).__name__}: {exc}')
        raise
    recovery_logs = []
    for path in session_root.glob('seed42_topology_launcher_*.log'):
        _checked_existing_path(path, directory=False)
        recovery_logs.append(path)
    record = {
        'status': 'recovered_incomplete_export_only',
        'run_directory': str(run_dir),
        'manifest': str(manifest_path),
        'manifest_status': manifest['status'],
        'reported_process_peak': 'approximately 551 MB; exact byte count not available in the incident summary',
        'configured_process_peak_limit_bytes': 2 * 1024**3,
        'checkpoint_expected': False,
        'launcher_logs_found': [str(path) for path in recovery_logs],
        'recovery_journal': str(journal_path),
        'diagnosis': 'The trainer writes incomplete before training, appends history after completed epochs, and catches Python exceptions to publish failed. Abrupt process/runtime termination before that handler can leave this state. The available record does not identify the cause.',
        'retry': 'prohibited; fixed launcher namespace is already occupied',
    }
    archive_path = export_directory / f'h96_seed42_incomplete_recovery_{uuid.uuid4().hex}.zip'
    partial_path = archive_path.with_suffix('.partial')
    archive_names = {manifest_path: 'run/run.json', journal_path: 'recovery_progress.jsonl'}
    archive_names.update({path: f'launcher_logs/{index}_{path.name}' for index, path in enumerate(recovery_logs, start=1)})
    included = [manifest_path, *recovery_logs, journal_path]
    journal('archive_write_started', target=str(archive_path), included_count=len(included))
    record['included_files'] = [{'archive_path': archive_names[path], 'size_bytes': path.stat().st_size, 'sha256': _sha256_path(path)} for path in included]
    try:
        with partial_path.open('xb') as raw_archive:
            with zipfile.ZipFile(raw_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
                archive.write(manifest_path, 'run/run.json')
                archive.writestr('recovery_report.json', json.dumps(record, indent=2, sort_keys=True))
                for index, log_path in enumerate(recovery_logs, start=1):
                    archive.write(log_path, f'launcher_logs/{index}_{log_path.name}')
                archive.write(journal_path, 'recovery_progress.jsonl')
            raw_archive.flush()
            os.fsync(raw_archive.fileno())
        if archive_path.exists():
            raise FileExistsError(f'Refusing to overwrite recovery archive: {archive_path}')
        os.rename(partial_path, archive_path)
        journal('archive_published', archive=str(archive_path), archive_sha256=_sha256_path(archive_path))
    except BaseException as exc:
        journal('archive_write_failed', partial_archive=str(partial_path) if partial_path.exists() else None, error=f'{type(exc).__name__}: {exc}')
        raise
    return {'archive': str(archive_path), 'progress_journal': str(journal_path), 'record': record}

print('Recovery mode: export only; no arrays, checkpoints, or training are accessed.')

## Export the existing incomplete attempt

Run the next cell only in the same Colab runtime where the attempt directory is present. It refuses any state other than the reported manifest-only incomplete run. Authorize a Drive mount only if you want a persistent copy; otherwise use the browser download. The export does not alter the run directory.

In [ ]:
if not RUN_DIR.exists():
    raise FileNotFoundError(f'Original Colab runtime/output is unavailable at {RUN_DIR}; use an already exported attempt archive if one exists. This notebook cannot recover lost /content files.')
persistent = input('Use Google Drive for persistent export? Type YES to mount, otherwise type DOWNLOAD: ').strip()
if persistent == 'YES':
    if input('Confirm Drive mount by typing MOUNT DRIVE FOR RECOVERY EXPORT: ').strip() != 'MOUNT DRIVE FOR RECOVERY EXPORT':
        raise RuntimeError('Drive mount was not explicitly authorized')
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    export_text = input('Exact existing Drive export directory: ').strip()
    export_dir = _checked_existing_path(Path(export_text).expanduser(), directory=True)
    download_fn = None
elif persistent == 'DOWNLOAD':
    from google.colab import files
    export_dir = _checked_existing_path(SESSION_ROOT, directory=True)
    download_fn = files.download
else:
    raise RuntimeError('Choose exactly YES or DOWNLOAD')
try:
    RECOVERY_RESULT = _export_incomplete_attempt(RUN_DIR, SESSION_ROOT, export_dir)
    print(json.dumps(RECOVERY_RESULT, indent=2, sort_keys=True))
except BaseException:
    print('Recovery stopped; inspect any h96_seed42_recovery_progress_*.jsonl file in the chosen export directory. The original attempt remains untouched.')
    raise
if download_fn is not None:
    download_fn(RECOVERY_RESULT['archive'])
print('Export complete. Original run directory was not modified; no retry was attempted.')